# 1. Creating Table

## 1.1 Adding Constraints

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cpt_utility_catalog.gold.fct_service_requests(
    service_key BIGINT NOT NULL,
    suburb_key BIGINT NOT NULL,
    created_date_key INT NOT NULL,
    changed_date_key INT,
    completed_date_key INT,
    notification STRING,
    work_center STRING,
    service_name STRING,

    CONSTRAINT pk_fct_service_requests PRIMARY KEY(service_key) RELY,
    CONSTRAINT fk_fct_service_requests_suburb FOREIGN KEY(suburb_key) REFERENCES cpt_utility_catalog.gold.dim_suburb(suburb_key) RELY,
    CONSTRAINT fk_fct_service_requests_created_date FOREIGN KEY(created_date_key) REFERENCES cpt_utility_catalog.gold.dim_date(date_key) RELY,
    CONSTRAINT fk_fct_service_requests_changed_date FOREIGN KEY(changed_date_key) REFERENCES cpt_utility_catalog.gold.dim_date(date_key) RELY,
    CONSTRAINT fk_fct_service_requests_completed_date FOREIGN KEY(completed_date_key) REFERENCES cpt_utility_catalog.gold.dim_date(date_key) RELY
)

## 1.2 Populating Table

In [0]:
%sql
INSERT OVERWRITE TABLE cpt_utility_catalog.gold.fct_service_requests
SELECT
xxhash64(s.id) AS service_key,

COALESCE(d.suburb_key, xxhash64('unmapped')) AS suburb_key,
COALESCE(CAST(date_format(s.created_on_date, 'yyyyMMdd') AS INT), -1) AS created_date_key,
CAST(date_format(s.changed_on_date, 'yyyyMMdd') AS INT) AS changed_date_key,
CAST(date_format(s.completed_on_date, 'yyyyMMdd') AS INT) AS completed_date_key,

s.notification AS notification,
s.work_center AS work_center,
s.c3_complaint_type AS service_name

FROM cpt_utility_catalog.silver.silver_service_requests_cleaned s
LEFT JOIN cpt_utility_catalog.gold.dim_suburb d
ON xxhash64(LOWER(TRIM(s.suburb))) = d.suburb_key